In [ ]:
import pandas as pd 
import numpy as np
import os
from dotenv import load_dotenv
import requests
import concurrent.futures
import csv
import time
from urllib.parse import quote
from tqdm import tqdm
load_dotenv()

In [ ]:
path = "data/raw/all_conferences_papers.csv"

In [ ]:
df[df['venue']=='KI']

In [ ]:
MY_EMAIL = ""#os.getenv("MY_EMAIL")
API_KEY = ""#os.getenv("API_KEY")

In [ ]:
OUTPUT_FILE = 'data/raw/openalex_results_KI_2020.csv' 
MAX_WORKERS = 10

In [ ]:
def reconstruct_abstract(inverted_index):
    if not inverted_index: return ""
    word_positions = []
    for word, positions in inverted_index.items():
        for pos in positions:
            word_positions.append((pos, word))
    word_positions.sort()
    return " ".join([word for _, word in word_positions])

In [ ]:
def get_data_authenticated(row):
    title = row.get('title')
    year = row.get('year')
    idx = row.name 

    if pd.isna(title):
        return None

    query_title = quote(str(title))
    
    url = (f"https://api.openalex.org/works?search={query_title}"
           f"&filter=publication_year:{int(year)}"
           f"&select=id,display_name,keywords,concepts,abstract_inverted_index")

    headers = {
        'User-Agent': f"mailto:{MY_EMAIL}",
        'Authorization': f"Bearer {API_KEY}"
    }

    empty_result = {
        'original_index': idx, 'oa_id': 'ERROR', 
        'author_keywords': '', 'ai_concepts': '', 'abstract': ''
    }

    try:
        response = requests.get(url, headers=headers, timeout=15)
        
        if response.status_code == 200:
            data = response.json()
            results = data.get('results', [])
            
            if results:
                top_hit = results[0]
                
                raw_keywords = top_hit.get('keywords', [])
                author_keywords_str = "; ".join([k['display_name'] for k in raw_keywords])

                raw_concepts = top_hit.get('concepts', [])
                filtered_concepts = []
                
                for c in raw_concepts:
                    if c['level'] < 2: continue
                    if c['score'] < 0.6: continue
                    
                    filtered_concepts.append(c['display_name'])

                if not filtered_concepts and raw_concepts:
                     fallback = [c['display_name'] for c in raw_concepts if c['level'] >= 1]
                     filtered_concepts = fallback[:3]

                ai_concepts_str = "; ".join(filtered_concepts)

                abstract_index = top_hit.get('abstract_inverted_index')
                abstract_text = reconstruct_abstract(abstract_index)

                return {
                    'original_index': idx,
                    'oa_id': top_hit.get('id'),
                    'author_keywords': author_keywords_str, # Spalte 1
                    'ai_concepts': ai_concepts_str,         # Spalte 2
                    'abstract': abstract_text
                }
            else:
                empty_result['oa_id'] = 'NOT_FOUND'
                return empty_result
        
        elif response.status_code == 429:
            time.sleep(1)
            empty_result['oa_id'] = 'RATE_LIMIT'
            return empty_result
        else:
            empty_result['oa_id'] = f'ERROR_{response.status_code}'
            return empty_result

    except Exception:
        empty_result['oa_id'] = 'EXCEPTION'
        return empty_result

In [ ]:
processed_indices = set()
if os.path.exists(OUTPUT_FILE):
    try:
        existing = pd.read_csv(OUTPUT_FILE)
        processed_indices = set(existing['original_index'].unique())
    except: pass

rows_to_process = [row for index, row in df.iterrows() if index not in processed_indices]

with open(OUTPUT_FILE, mode='a', newline='', encoding='utf-8') as f:
    fieldnames = ['original_index', 'oa_id', 'author_keywords', 'ai_concepts', 'abstract']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    
    if not os.path.exists(OUTPUT_FILE) or os.path.getsize(OUTPUT_FILE) == 0:
        writer.writeheader()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_row = {executor.submit(get_data_authenticated, row): row for row in rows_to_process}
        
        for future in tqdm(concurrent.futures.as_completed(future_to_row), total=len(rows_to_process)):
            result = future.result()
            if result:
                writer.writerow(result)
                f.flush()

print("Finished Saved in:", OUTPUT_FILE)